# EXP23: 近期 vs 历史表现对比

策略是否在近年（2023-2026）表现与早期（2015-2019）不同？
金价波动特征是否发生结构性变化？

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Ready')

In [ ]:
# Part 1: 分时期回测
periods = [
    ("2015-2017", "2015-01-01", "2018-01-01"),
    ("2018-2019", "2018-01-01", "2020-01-01"),
    ("2020-2021", "2020-01-01", "2022-01-01"),
    ("2022-2023", "2022-01-01", "2024-01-01"),
    ("2024-2026", "2024-01-01", "2026-04-01"),
]

period_results = []
for name, start, end in periods:
    d = data.slice(start, end)
    r = run_variant(d, name, **LIVE_KWARGS)
    
    trades = r['_trades']
    avg_atr = d.h1_df['ATR'].mean() if 'ATR' in d.h1_df.columns else 0
    r['period'] = name
    r['avg_atr'] = avg_atr
    r['per_trade'] = r['total_pnl'] / r['n'] if r['n'] > 0 else 0
    
    kc_trades = [t for t in trades if t.strategy == 'keltner']
    orb_trades = [t for t in trades if t.strategy == 'orb']
    r['kc_pnl'] = sum(t.pnl for t in kc_trades)
    r['orb_pnl'] = sum(t.pnl for t in orb_trades)
    r['kc_n'] = len(kc_trades)
    r['orb_n'] = len(orb_trades)
    
    period_results.append(r)
    print(f"{name}: Sharpe={r['sharpe']:.2f}, PnL=${r['total_pnl']:.0f}, N={r['n']}, "
          f"$/trade=${r['per_trade']:.2f}, ATR={avg_atr:.2f}")

In [ ]:
# Part 2: 近期 vs 早期对比表
df = pd.DataFrame([{
    'period': r['period'], 'sharpe': r['sharpe'], 'pnl': r['total_pnl'],
    'n': r['n'], 'per_trade': r['per_trade'], 'avg_atr': r['avg_atr'],
    'kc_pnl': r['kc_pnl'], 'orb_pnl': r['orb_pnl'],
    'kc_n': r['kc_n'], 'orb_n': r['orb_n'],
} for r in period_results])
print(df.to_string(index=False))

# ATR 趋势
print(f"\nATR trend: {' -> '.join(f'{r:.2f}' for r in df['avg_atr'])}")
print(f"$/trade trend: {' -> '.join(f'${r:.2f}' for r in df['per_trade'])}")

In [ ]:
# Part 3: 滚动 6 个月 Sharpe
full = run_variant(data, "Full", **LIVE_KWARGS)
trades = full['_trades']

monthly = pd.DataFrame([{
    'month': t.entry_time.strftime('%Y-%m'),
    'pnl': t.pnl,
} for t in trades])
monthly_pnl = monthly.groupby('month')['pnl'].sum()

rolling_sharpe = monthly_pnl.rolling(6).apply(
    lambda x: x.mean() / x.std() * np.sqrt(12) if x.std() > 0 else 0
)

print("\n=== Rolling 6-Month Sharpe ===")
for m, sh in rolling_sharpe.dropna().items():
    bar = '+' * max(0, int(sh)) + '-' * max(0, int(-sh))
    print(f"  {m}: {sh:6.2f}  {bar}")

In [ ]:
import json
from backtest.runner import sanitize_for_json
with open('../data/exp23_results.json', 'w') as f:
    json.dump(sanitize_for_json(period_results), f, indent=2)
print('Saved')